# X-probing

With PyLabRobot, you can probe the x position of any conductive object on a STAR(let) deck using capacitive liquid level detection (cLLD). The channel is moved laterally along the x-axis until the cLLD sensor detects a change in capacitance, and the x-coordinate of the material boundary is returned.

See also [y-probing](./y) and [z-probing](./z) for probing in the other directions.

```{warning}
This example uses the teaching tips. These are metal tips that are not forgiving. Be particularly careful when moving the channels around to avoid collisions.
```

## Setup

Import {class}`~pylabrobot.hamilton.liquid_handlers.star.star.STAR` and one of the Hamilton deck classes.

In [ ]:
from pylabrobot.hamilton.liquid_handlers.star import STAR

star = STAR()
await star.setup()

## Capacitive x-probing using cLLD

X-probing uses the cLLD sensor to detect conductive surfaces by moving the channel laterally along the x-axis. Starting from the channel's current x position, the channel moves in the specified direction (`"right"` or `"left"`) until cLLD triggers. The returned value is a geometric estimate of the material boundary, corrected by half the tip bottom diameter.

```{warning}
This example uses the teaching tips. These are metal tips that are not forgiving. Be particularly careful when moving the channels around to avoid collisions. Make sure the tip is at a safe z-height before initiating lateral x movement.
```

### Probing a single point

Pick up a teaching tip:

In [ ]:
teaching_tip_rack = star.deck.get_resource("teaching_tip_rack")
await star.pip.pick_up_tips(teaching_tip_rack["A1"])

Prepare channel 0 for manual operation and move it to the desired y/z position. The channel must be at a z-height where lateral x movement is safe (i.e. the tip is at the same height as the surface to detect).

For more information on manually moving channels, see [Manually moving channels around](../../00_liquid-handling/moving-channels-around.ipynb).

In [3]:
await star.driver.pip.prepare_for_manual_channel_operation(0)

# TODO: change these to positions that work for you
await star.driver.left_x_arm.move_to(500)
await star.driver.pip.move_channel_y(0, 300)

Use `clld_probe_x_position` to probe the x-position of a conductive surface. The channel moves laterally in the specified direction until the cLLD sensor triggers. The estimated x-coordinate of the material boundary is returned, corrected by half the tip bottom diameter.

You can probe in both directions:
- `"right"`: the channel moves toward higher x values (away from the instrument origin)
- `"left"`: the channel moves toward lower x values (toward the instrument origin)

Note: x-probing is quite slow so it might appear that is not moving.

In [4]:
await star.driver.left_x_arm.clld_probe_x_position(
  channel_idx=0,
  probing_direction="right",
)

501.2

In [ ]:
await star.driver.left_x_arm.clld_probe_x_position(
  channel_idx=0,
  probing_direction="left",
)

### Optional parameters

`clld_probe_x_position` accepts several optional parameters:

- `end_pos_search` (float, mm): The x-position at which to stop searching. Defaults to the instrument's reachable limit in the probing direction.
- `post_detection_dist` (float, mm): Distance to retract after detection. Defaults to `2.0` mm.
- `tip_bottom_diameter` (float, mm): Diameter of the tip bottom, used for geometric correction. Defaults to `1.2` mm (teaching needle).
- `read_timeout` (float, seconds): Timeout for the probing command. Defaults to `240.0` seconds.

In [ ]:
await star.driver.left_x_arm.clld_probe_x_position(
  channel_idx=0,
  probing_direction="right",
  end_pos_search=600,
  post_detection_dist=3.0,
)

Return the teaching tip when done:

In [5]:
await star.pip.return_tips()

## Teardown

In [ ]:
await star.stop()